## Zonal Stats for the Specific sites along Lake Huron

# The following I create a Watershed for each site 

In [11]:
#reqired imports
import os
import arcpy
from arcpy import env
from arcpy.sa import SnapPourPoint, Watershed
from arcpy import sa
import numpy as np
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")
arcpy.env.addOutputsToMap = False  # helps avoid schema lock

# Inputs 

inDir = r"D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer"

CW_path = r"D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites"

Coastalwetlands_original = os.path.join(CW_path, "test_aggregation_polys.shp")

inStreams = os.path.join(inDir, "GLB_Stream", "GLB_stream_Ras_FeatureToLine.shp")
D8_flow   = r"S:\Projects\Active\GLB_Nutrient_Transport\DEM_rasters\GLB_Bdry_buff10km_dem_fill_dir.tif"
flowacc = r"S:\Projects\Active\GLB_Nutrient_Transport\DEM_rasters\GLB_Bdry_buff10km_dem_fill_flowaccu.tif"
inStreamsWatershed = os.path.join(inDir, "Streamwatershed", "PointWaterdhed_LH.shp")
Lake_Huron = r"D:\Users\abolmaal\code\boundry\hydro_p_LakeHuron\hydro_p_LakeHuron.shp"
# -------------------------------------------------------------------
# Parameters / field names
# -------------------------------------------------------------------
CW_ID_FIELD = "CW_Id"          # stable wetland id
COASTAL_ID_FIELD = "Coastal_Id" # optional (we'll set equal to CW_Id unless you want different)
COASTAL_Area_FIELD = "CW_Aream2"  # area of coastal watershed in m2
Wetland_Area_FIELD = "WS_Aream2"  # area of wetland in m2
crs_Albers = arcpy.SpatialReference(3174)  # Great Lakes Albers meters
crs_WGS84  = arcpy.SpatialReference(4326)  # WGS84
# -------------------------------------------------------------------
outpath= os.path.join(inDir, "CW_path", "CoastalWatersheds")
outErase_Riper   = os.path.join(outpath, "Erase_Riperian")
outErase_drainage= os.path.join(outpath, "Erase_drainage")
outErase_Lake    = os.path.join(outpath, "Erase_lake")
outPourpoints    = os.path.join(outpath, "Pourpoints")
outWatersheds    = os.path.join(outpath, "Watershed_rasters")

for d in [outpath, outErase_Riper, outErase_drainage, outErase_Lake, outPourpoints, outWatersheds]:
    os.makedirs(d, exist_ok=True)

erase_buffer   = os.path.join(outErase_Riper, "Wetland_connected_erasebuff_50.shp")
Coastalwatershed_original = os.path.join(outpath, "test_coastal_watersheds.shp")

# Helpers

In [2]:
def convert_kgday_to_grm2yr(df, kgday_col, area_col, new_col):
    """
    Converts values from kg/day to grams/m²/year.

    Parameters:
    - df: DataFrame containing the data
    - kgday_col: Column with kg/day values
    - area_col: Column with area in m²
    - new_col: Name of the output column (g/m²/year)
    """
    df[new_col] = ((df[kgday_col] * 1000 * 365) / df[area_col])

In [4]:
# -------------------------------------------------------------------
# Parameters / field names
# -------------------------------------------------------------------
CW_ID_FIELD        = "CW_Id"            # stable wetland id
COASTAL_ID_FIELD   = "Coastal_Id"       # optional (set equal to CW_Id here)
COASTAL_AREA_FIELD = "CW_Aream2"  # (here: wetland polygon area in m²)

wetlands_fcs = [
    Coastalwetlands_original,
]

for fc in wetlands_fcs:

    # 1) Ensure fields exist (adds them if missing)
    ensure_field(fc, CW_ID_FIELD, "LONG")
    ensure_field(fc, COASTAL_ID_FIELD, "LONG")
    ensure_field(fc, COASTAL_AREA_FIELD, "DOUBLE")

    # 2) Re-read fields AFTER ensuring (important!)
    fields = [f.name for f in arcpy.ListFields(fc)]

    # 3) Populate CW_Id and Coastal_Id
    if "Id" in fields:
        arcpy.management.CalculateField(fc, CW_ID_FIELD, "!Id!", "PYTHON3")
        arcpy.management.CalculateField(fc, COASTAL_ID_FIELD, "!Id!", "PYTHON3")
    else:
        oid = arcpy.Describe(fc).OIDFieldName
        arcpy.management.CalculateField(fc, CW_ID_FIELD, f"!{oid}!", "PYTHON3")
        arcpy.management.CalculateField(fc, COASTAL_ID_FIELD, f"!{oid}!", "PYTHON3")

    # 4) Compute polygon area (m²) for THIS layer (wetland area)
    #    This uses the dataset CRS units. If fc is EPSG:3174, area will be m².
    try:
        arcpy.management.CalculateGeometryAttributes(
            fc,
            [[COASTAL_AREA_FIELD, "AREA"]],
            area_unit="SQUARE_METERS"
        )
    except Exception as e:
        print(f"⚠️ Could not calculate {COASTAL_AREA_FIELD} for {os.path.basename(fc)}: {e}")

    print(f"✅ ensured fields + IDs + area on: {os.path.basename(fc)}")




✅ ensured fields + IDs + area on: test_aggregation_polys.shp


In [3]:
# ============================================================
# Watersheds from Coastalwetlands_original
# - Adds CW_Id on copy (needed for SnapPourPoint / Watershed IDs)
# - Adds centroid fields:
#     Wetlands:  CW_cx, CW_cy
#     Watersheds: WS_cx, WS_cy
# - Adds WS_AREAM2 (m²) on final watershed shapefile
# - NO stream erase
# - ALL outputs saved under CW_path
# ============================================================

import os
import arcpy
from arcpy import sa

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
arcpy.CheckOutExtension("Spatial")

# -----------------------------
# Inputs
# -----------------------------
CW_path = r"D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites"
Coastalwetlands_original = os.path.join(CW_path, "test_aggregation_polys.shp")

D8_flow = r"S:\Projects\Active\GLB_Nutrient_Transport\DEM_rasters\GLB_Bdry_buff10km_dem_fill_dir.tif"
flowacc = r"S:\Projects\Active\GLB_Nutrient_Transport\DEM_rasters\GLB_Bdry_buff10km_dem_fill_flowaccu.tif"

# -----------------------------
# Outputs (ALL under CW_path)
# -----------------------------
out_gdb = os.path.join(CW_path, "cw_watersheds.gdb")
if not arcpy.Exists(out_gdb):
    arcpy.management.CreateFileGDB(CW_path, "cw_watersheds.gdb")

# output shapefiles in CW_path
out_wetlands_shp   = os.path.join(CW_path, "test_coastalwetlands_with_cxy.shp")
out_pourpoints_shp = os.path.join(CW_path, "test_pourpoints_snapped.shp")
out_watersheds_shp = os.path.join(CW_path, "test_coastal_watersheds.shp")

# intermediate in GDB
wet_copy   = os.path.join(out_gdb, "wet_copy")
wet_proj   = os.path.join(out_gdb, "wet_proj")
pp_raw     = os.path.join(out_gdb, "pp_raw")
pp_sn_ras  = os.path.join(out_gdb, "pp_snapped_ras")
ws_ras     = os.path.join(out_gdb, "ws_raster")
ws_polyraw = os.path.join(out_gdb, "ws_poly_raw")
ws_polyd   = os.path.join(out_gdb, "ws_poly_diss")

CW_ID_FIELD   = "CW_Id"
WS_AREA_FIELD = "WS_AREAM2"

# centroid fields
WET_CX, WET_CY = "CW_cx", "CW_cy"
WS_CX,  WS_CY  = "WS_cx", "WS_cy"

# -----------------------------
# Environment: align to D8 grid
# -----------------------------
D8_SR = arcpy.Describe(D8_flow).spatialReference
arcpy.env.workspace = out_gdb
arcpy.env.scratchWorkspace = out_gdb
arcpy.env.snapRaster = D8_flow
arcpy.env.cellSize = D8_flow
arcpy.env.extent = D8_flow
arcpy.env.outputCoordinateSystem = D8_SR

cellsize = float(arcpy.Describe(D8_flow).meanCellWidth)

def safe_del(p):
    if p and arcpy.Exists(p):
        arcpy.management.Delete(p)

def ensure_field(fc, name, ftype):
    fields = [f.name for f in arcpy.ListFields(fc)]
    if name not in fields:
        arcpy.management.AddField(fc, name, ftype)
    return name

def calc_area_m2(fc, out_field):
    ensure_field(fc, out_field, "DOUBLE")
    arcpy.management.CalculateGeometryAttributes(
        fc, [[out_field, "AREA"]],
        area_unit="SQUARE_METERS"
    )

def add_centroid_xy(fc, x_field, y_field, coord_sr):
    """
    Adds centroid X/Y in coord_sr.
    If fc is already in coord_sr, this is just centroid coords.
    """
    ensure_field(fc, x_field, "DOUBLE")
    ensure_field(fc, y_field, "DOUBLE")
    try:
        arcpy.management.CalculateGeometryAttributes(
            fc,
            [[x_field, "CENTROID_X"], [y_field, "CENTROID_Y"]],
            coordinate_system=coord_sr
        )
    except Exception:
        # fallback
        arcpy.management.CalculateField(fc, x_field, "!SHAPE.centroid.X!", "PYTHON3")
        arcpy.management.CalculateField(fc, y_field, "!SHAPE.centroid.Y!", "PYTHON3")

# -----------------------------
# 1) Copy wetlands into GDB (don’t touch original)
# -----------------------------
safe_del(wet_copy)
arcpy.management.CopyFeatures(Coastalwetlands_original, wet_copy)

# If SR is unknown on the copy, define it as D8 SR (define only; no coordinate move)
wet_sr = arcpy.Describe(wet_copy).spatialReference
if (wet_sr is None) or (wet_sr.name in ("Unknown", "", None)) or (getattr(wet_sr, "factoryCode", 0) in (0, None)):
    print("⚠️ Wetlands SR is Unknown. Defining projection on COPY to match D8_flow SR.")
    arcpy.management.DefineProjection(wet_copy, D8_SR)

# -----------------------------
# 2) Project wetlands to D8 SR (if needed)
# -----------------------------
safe_del(wet_proj)
wet_sr2 = arcpy.Describe(wet_copy).spatialReference
if wet_sr2 and getattr(wet_sr2, "factoryCode", None) == getattr(D8_SR, "factoryCode", None):
    arcpy.management.CopyFeatures(wet_copy, wet_proj)
else:
    # optional transform if available
    try:
        tx = arcpy.ListTransformations(wet_sr2, D8_SR)
        transform = tx[0] if tx else None
    except Exception:
        transform = None

    if transform:
        arcpy.management.Project(wet_copy, wet_proj, D8_SR, transform)
    else:
        arcpy.management.Project(wet_copy, wet_proj, D8_SR)

arcpy.management.RepairGeometry(wet_proj)

# -----------------------------
# 3) Ensure CW_Id exists on wet_proj (COPY ONLY)
# -----------------------------
ensure_field(wet_proj, CW_ID_FIELD, "LONG")
wet_fields = [f.name for f in arcpy.ListFields(wet_proj)]
oid_field = arcpy.Describe(wet_proj).OIDFieldName

if "Id" in wet_fields:
    arcpy.management.CalculateField(wet_proj, CW_ID_FIELD, "!Id!", "PYTHON3")
    print("ℹ️ Set CW_Id = Id")
else:
    arcpy.management.CalculateField(wet_proj, CW_ID_FIELD, f"!{oid_field}!", "PYTHON3")
    print(f"ℹ️ Set CW_Id = {oid_field}")

# -----------------------------
# 4) Add wetlands centroid fields (CW_cx, CW_cy) on the COPY
# -----------------------------
add_centroid_xy(wet_proj, WET_CX, WET_CY, D8_SR)

# Save wetlands with centroids to CW_path (so you can use it later)
safe_del(out_wetlands_shp)
arcpy.management.CopyFeatures(wet_proj, out_wetlands_shp)

# -----------------------------
# 5) Pour points (one per polygon feature) - carries CW_Id
# -----------------------------
safe_del(pp_raw)
arcpy.management.FeatureToPoint(wet_proj, pp_raw, "INSIDE")

# -----------------------------
# 6) SnapPourPoint (raster values = CW_Id)
# -----------------------------
safe_del(pp_sn_ras)
snapped = sa.SnapPourPoint(pp_raw, flowacc, 200, CW_ID_FIELD)  # 200 meters snap
snapped.save(pp_sn_ras)

# Export snapped pour points for inspection
safe_del(out_pourpoints_shp)
arcpy.conversion.RasterToPoint(pp_sn_ras, out_pourpoints_shp, "VALUE")
ensure_field(out_pourpoints_shp, CW_ID_FIELD, "LONG")
arcpy.management.CalculateField(out_pourpoints_shp, CW_ID_FIELD, "!grid_code!", "PYTHON3")

# -----------------------------
# 7) Watershed raster
# -----------------------------
safe_del(ws_ras)
sa.Watershed(D8_flow, pp_sn_ras).save(ws_ras)

# -----------------------------
# 8) Raster to polygon + CW_Id + dissolve by CW_Id
# -----------------------------
safe_del(ws_polyraw)
arcpy.conversion.RasterToPolygon(ws_ras, ws_polyraw, "NO_SIMPLIFY", "VALUE")

ensure_field(ws_polyraw, CW_ID_FIELD, "LONG")
arcpy.management.CalculateField(ws_polyraw, CW_ID_FIELD, "!gridcode!", "PYTHON3")

safe_del(ws_polyd)
if hasattr(arcpy.analysis, "PairwiseDissolve"):
    arcpy.analysis.PairwiseDissolve(ws_polyraw, ws_polyd, CW_ID_FIELD)
else:
    arcpy.management.Dissolve(ws_polyraw, ws_polyd, CW_ID_FIELD)

# -----------------------------
# 9) Save final watersheds to SHP + calculate WS_AREAM2 + WS_cx/WS_cy
# -----------------------------
safe_del(out_watersheds_shp)
arcpy.management.CopyFeatures(ws_polyd, out_watersheds_shp)

calc_area_m2(out_watersheds_shp, WS_AREA_FIELD)
add_centroid_xy(out_watersheds_shp, WS_CX, WS_CY, D8_SR)

print("✅ DONE")
print(f"✅ Wetlands w/ CW_Id + CW_cx/CW_cy: {out_wetlands_shp}")
print(f"✅ Snapped pourpoints: {out_pourpoints_shp}")
print(f"✅ Final watersheds w/ CW_Id + WS_AREAM2 + WS_cx/WS_cy: {out_watersheds_shp}")
print(f"✅ Intermediates in: {out_gdb}")


ℹ️ Set CW_Id = OBJECTID
✅ DONE
✅ Wetlands w/ CW_Id + CW_cx/CW_cy: D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites\test_coastalwetlands_with_cxy.shp
✅ Snapped pourpoints: D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites\test_pourpoints_snapped.shp
✅ Final watersheds w/ CW_Id + WS_AREAM2 + WS_cx/WS_cy: D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites\test_coastal_watersheds.shp
✅ Intermediates in: D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites\cw_watersheds.gdb


# Estimate zonal States for each watershed

In [4]:
# Parameters for reading shapefiles
# define the coastal wetland number which is same as FID number+1
Coastal_num = "CW_Id"
# coastal id is a unique number that is equal to = ID * 1000 + Coastal_num
Coastal_ID = 'Coastal_Id'
# coastal wetland fields
CoastalWetland_Area = "CW_Aream2"
lat_col_CW ="CW_cx"
lon_col_CW ="CW_cy"
# coastal watershed fields 
CoastalWatershed_Area = "WS_AREAM2"
lat_col_W ="WS_cx"
lon_col_W ="WS_cy"

# Fields to calculate 
# Direct delivery to Coastal Watersheds
CoastalDirectTN = 'CoastalWatershedDirectTN_kgday'
CoastalDirectTN_convert = 'CoastalWatershedDirectTN_grm2yr'
CoastalDirectTP = 'CoastalWatershedDirectTP_kgday'
CoastalDirectTP_convert = 'CoastalWatershedDirectTP_grm2yr'

# out names to save the zonal stats for the coastal watersheds
outCoastalWatershed = os.path.join(CW_path, 'DirectTNTP_CoastalWatershed.csv')

In [ ]:
import os
import arcpy
import pandas as pd
from arcpy import sa

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")
# -----------------------------
# Inputs
# -----------------------------
CW_path = r"D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites"
InputWatersheds = os.path.join(CW_path, "test_coastal_watersheds.shp")

base_path = r"D:\Users\abolmaal\Arcgis\NASAOceanProject"
nutrient_flux_path = os.path.join(base_path, "Luwen_Nutrient")
inBYRaster_TN = os.path.join(nutrient_flux_path, "TN_Annual_delTotal_header_kgcellday.tif")
inBYRaster_TP = os.path.join(nutrient_flux_path, "TP_Annual_delTotal_header_kgcellday.tif")

In [ ]:
import os
import arcpy
import pandas as pd
from arcpy import sa

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# -----------------------------
# Inputs
# -----------------------------
CW_path = r"D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites"
InputWatersheds = os.path.join(CW_path, "test_coastal_watersheds.shp")

base_path = r"D:\Users\abolmaal\Arcgis\NASAOceanProject"
nutrient_flux_path = os.path.join(base_path, "Luwen_Nutrient")
inBYRaster_TN = os.path.join(nutrient_flux_path, "TN_Annual_delTotal_header_kgcellday.tif")
inBYRaster_TP = os.path.join(nutrient_flux_path, "TP_Annual_delTotal_header_kgcellday.tif")

# -----------------------------
# Fields (from your watershed output)
# -----------------------------
Coastal_num = "CW_Id"
CoastalWatershed_Area = "WS_AREAM2"
x_col_W = "WS_cx"   # centroid X in EPSG:3174 (meters)
y_col_W = "WS_cy"   # centroid Y in EPSG:3174 (meters)

# Output columns
CoastalDirectTN = "CoastalWatershedDirectTN_kgday"
CoastalDirectTP = "CoastalWatershedDirectTP_kgday"

# Save CSV
outCoastalWatershed = os.path.join(CW_path, "DirectTNTP_CoastalWatershed.csv")

# -----------------------------
# 1) Read watershed attributes (this part is fine)
# -----------------------------
dfCoastalWatershed = pd.DataFrame(
    arcpy.da.TableToNumPyArray(
        InputWatersheds,
        [Coastal_num, CoastalWatershed_Area, x_col_W, y_col_W]
    )
)

# -----------------------------
# 2) Zonal stats MUST use the feature class, not the DataFrame
# -----------------------------
ztn = "in_memory\\ztn"
ztp = "in_memory\\ztp"
if arcpy.Exists(ztn): arcpy.management.Delete(ztn)
if arcpy.Exists(ztp): arcpy.management.Delete(ztp)

inZonalTN_coastal = sa.ZonalStatisticsAsTable(
    in_zone_data=InputWatersheds,
    zone_field=Coastal_num,
    in_value_raster=inBYRaster_TN,
    out_table=ztn,
    ignore_nodata="DATA",
    statistics_type="SUM"
)

inZonalTP_coastal = sa.ZonalStatisticsAsTable(
    in_zone_data=InputWatersheds,
    zone_field=Coastal_num,
    in_value_raster=inBYRaster_TP,
    out_table=ztp,
    ignore_nodata="DATA",
    statistics_type="SUM"
)

# -----------------------------
# 3) Read zonal outputs + rename SUM columns
# -----------------------------
dfZonalTN = pd.DataFrame(arcpy.da.TableToNumPyArray(ztn, [Coastal_num, "SUM"]))
dfZonalTP = pd.DataFrame(arcpy.da.TableToNumPyArray(ztp, [Coastal_num, "SUM"]))

dfZonalTN.rename(columns={"SUM": CoastalDirectTN}, inplace=True)
dfZonalTP.rename(columns={"SUM": CoastalDirectTP}, inplace=True)

# -----------------------------
# 4) Merge and save
# -----------------------------
dfCoastalWatershed = dfCoastalWatershed.merge(dfZonalTN, on=Coastal_num, how="left")
dfCoastalWatershed = dfCoastalWatershed.merge(dfZonalTP, on=Coastal_num, how="left")

dfCoastalWatershed.to_csv(outCoastalWatershed, index=False)
print(f"✅ Saved: {outCoastalWatershed}")


In [11]:
import os
import numpy as np
import pandas as pd
import arcpy
from arcpy import sa

def zonal_stats_direct_TNTP_normalized_by_wetland_area(
    CW_path,
    watersheds_fc,          # e.g., os.path.join(CW_path, "test_coastal_watersheds.shp")
    wetlands_fc,            # e.g., os.path.join(CW_path, "test_coastalwetlands_with_cxy.shp")
    tn_raster,              # kg/cell/day raster
    tp_raster,              # kg/cell/day raster
    out_csv_name="DirectTNTP_CoastalWatershed.csv",
    id_field="CW_Id",

    # watershed fields (kept for reference/output)
    ws_area_field="WS_AREAM2",
    ws_x_field="WS_cx",
    ws_y_field="WS_cy",

    # wetland area field (USED for normalization)
    wet_area_field="CW_Aream2",
    calculate_wet_area_if_missing=True,

    # output kg/day column names
    tn_kgday_col="CoastalWatershedDirectTN_kgday",
    tp_kgday_col="CoastalWatershedDirectTP_kgday",

    # conversion options
    days_per_year=365,
    ignore_nodata="DATA"    # "DATA" or "NODATA"
):
    """
    Zonal stats SUM on watersheds (kg/day), then normalize using wetland area (CW_Aream2):
      - kg/day -> g/m²/day  (divide by CW_Aream2)
      - kg/day -> g/m²/yr   (divide by CW_Aream2 * and multiply by days/year)

    Returns the output DataFrame.
    """

    arcpy.env.overwriteOutput = True
    arcpy.env.addOutputsToMap = False
    arcpy.CheckOutExtension("Spatial")

    # -----------------------------
    # helpers
    # -----------------------------
    def _safe_delete(p):
        try:
            if p and arcpy.Exists(p):
                arcpy.management.Delete(p)
        except Exception:
            pass

    def _require_fields(fc, fields):
        existing = {f.name for f in arcpy.ListFields(fc)}
        missing = [f for f in fields if f not in existing]
        if missing:
            raise ValueError(f"Missing required field(s) in {fc}: {missing}")

    def ensure_field(fc, name, ftype):
        existing = {f.name for f in arcpy.ListFields(fc)}
        if name not in existing:
            arcpy.management.AddField(fc, name, ftype)
        return name

    def calc_area_m2(fc, out_field):
        ensure_field(fc, out_field, "DOUBLE")
        arcpy.management.CalculateGeometryAttributes(
            fc, [[out_field, "AREA"]],
            area_unit="SQUARE_METERS"
        )

    def convert_kgday_to_grm2day(df, kgday_col, area_col, new_col):
        area = df[area_col].astype(float)
        val  = df[kgday_col].astype(float)
        df[new_col] = np.where(
            (area > 0) & np.isfinite(area) & np.isfinite(val),
            (val * 1000.0) / area,
            np.nan
        )

    def convert_kgday_to_grm2yr(df, kgday_col, area_col, new_col, days_per_year):
        area = df[area_col].astype(float)
        val  = df[kgday_col].astype(float)
        df[new_col] = np.where(
            (area > 0) & np.isfinite(area) & np.isfinite(val),
            (val * 1000.0 * float(days_per_year)) / area,
            np.nan
        )

    # -----------------------------
    # validate inputs
    # -----------------------------
    if not arcpy.Exists(watersheds_fc):
        raise FileNotFoundError(f"Watersheds feature class not found: {watersheds_fc}")
    if not arcpy.Exists(wetlands_fc):
        raise FileNotFoundError(f"Wetlands feature class not found: {wetlands_fc}")
    if not arcpy.Exists(tn_raster):
        raise FileNotFoundError(f"TN raster not found: {tn_raster}")
    if not arcpy.Exists(tp_raster):
        raise FileNotFoundError(f"TP raster not found: {tp_raster}")

    _require_fields(watersheds_fc, [id_field, ws_area_field, ws_x_field, ws_y_field])
    _require_fields(wetlands_fc, [id_field])

    # Ensure wetland area exists (optionally calculate)
    wet_fields = {f.name for f in arcpy.ListFields(wetlands_fc)}
    if wet_area_field not in wet_fields:
        if calculate_wet_area_if_missing:
            print(f"⚠️ {wet_area_field} missing on wetlands -> calculating it (m²).")
            calc_area_m2(wetlands_fc, wet_area_field)
        else:
            raise ValueError(f"Wetland area field '{wet_area_field}' not found on {wetlands_fc}")

    # -----------------------------
    # 1) Base attribute table from watersheds (kept for context)
    # -----------------------------
    df_ws = pd.DataFrame(
        arcpy.da.TableToNumPyArray(
            watersheds_fc,
            [id_field, ws_area_field, ws_x_field, ws_y_field]
        )
    )

    # -----------------------------
    # 2) Wetland area table (CW_Aream2) and join by CW_Id
    # -----------------------------
    df_wet = pd.DataFrame(
        arcpy.da.TableToNumPyArray(
            wetlands_fc,
            [id_field, wet_area_field]
        )
    )

    # If wetlands have multiple parts per CW_Id, keep first non-null area
    df_wet = (
        df_wet.sort_values(by=[id_field])
              .dropna(subset=[wet_area_field])
              .drop_duplicates(subset=[id_field], keep="first")
    )

    df = df_ws.merge(df_wet, on=id_field, how="left")

    # -----------------------------
    # 3) Zonal stats SUM (TN/TP) on watersheds
    # -----------------------------
    ztn = "in_memory\\ztn_direct"
    ztp = "in_memory\\ztp_direct"
    _safe_delete(ztn)
    _safe_delete(ztp)

    sa.ZonalStatisticsAsTable(
        in_zone_data=watersheds_fc,
        zone_field=id_field,
        in_value_raster=tn_raster,
        out_table=ztn,
        ignore_nodata=ignore_nodata,
        statistics_type="SUM"
    )

    sa.ZonalStatisticsAsTable(
        in_zone_data=watersheds_fc,
        zone_field=id_field,
        in_value_raster=tp_raster,
        out_table=ztp,
        ignore_nodata=ignore_nodata,
        statistics_type="SUM"
    )

    df_tn = pd.DataFrame(arcpy.da.TableToNumPyArray(ztn, [id_field, "SUM"])).rename(columns={"SUM": tn_kgday_col})
    df_tp = pd.DataFrame(arcpy.da.TableToNumPyArray(ztp, [id_field, "SUM"])).rename(columns={"SUM": tp_kgday_col})

    df = df.merge(df_tn, on=id_field, how="left")
    df = df.merge(df_tp, on=id_field, how="left")

    # -----------------------------
    # 4) Conversions USING wetland area (CW_Aream2)
    # -----------------------------
    tn_grm2day_col = tn_kgday_col.replace("_kgday", "_grm2day")
    tp_grm2day_col = tp_kgday_col.replace("_kgday", "_grm2day")
    tn_grm2yr_col  = tn_kgday_col.replace("_kgday", "_grm2yr")
    tp_grm2yr_col  = tp_kgday_col.replace("_kgday", "_grm2yr")

    convert_kgday_to_grm2day(df, tn_kgday_col, wet_area_field, tn_grm2day_col)
    convert_kgday_to_grm2day(df, tp_kgday_col, wet_area_field, tp_grm2day_col)
    convert_kgday_to_grm2yr(df, tn_kgday_col, wet_area_field, tn_grm2yr_col, days_per_year)
    convert_kgday_to_grm2yr(df, tp_kgday_col, wet_area_field, tp_grm2yr_col, days_per_year)

    # -----------------------------
    # 5) Save CSV
    # -----------------------------
    out_csv = os.path.join(CW_path, out_csv_name)
    df.to_csv(out_csv, index=False)
    print(f"✅ Saved: {out_csv}")

    # quick warning if any CW_Id didn't find wetland area
    n_missing_area = int(df[wet_area_field].isna().sum())
    if n_missing_area > 0:
        print(f"⚠️ {n_missing_area} rows missing {wet_area_field} after join. Those conversions will be NaN.")

    return df


In [12]:
df_out = zonal_stats_direct_TNTP_normalized_by_wetland_area(
    CW_path=CW_path,
    watersheds_fc=os.path.join(CW_path, "test_coastal_watersheds.shp"),
    wetlands_fc=os.path.join(CW_path, "test_coastalwetlands_with_cxy.shp"),
    tn_raster=inBYRaster_TN,
    tp_raster=inBYRaster_TP,
    days_per_year=365
)

✅ Saved: D:\Users\abolmaal\Arcgis\NASAOceanProject\GIS_layer\chosensites\test_sites\DirectTNTP_CoastalWatershed.csv
